In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)

n = 1000

products = {
    "Phone": 15000,
    "Laptop": 65000,
    "Headphones": 2500,
    "Tablet": 28000,
    "Charger": 1200
}

sales = pd.DataFrame({
    "order_id": range(1001, 1001 + n),
    "order_date": pd.to_datetime("2026-01-01")
                  + pd.to_timedelta(rng.integers(0, 180, n), unit="D"),
    "branch": rng.choice(
        ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"],
        n,
        p=[0.4, 0.2, 0.15, 0.15, 0.1]
    ),
    "product": rng.choice(list(products), n),
    "quantity": rng.integers(1, 6, n),
    "channel": rng.choice(
        ["In-store", "Online", "Phone order"],
        n,
        p=[0.55, 0.35, 0.10]
    ),
})

sales["unit_price"] = sales["product"].map(products)

sales.to_csv("duka_sales.csv", index=False)

In [2]:
import os
os.path.exists("duka_sales.csv")

True

In [3]:
from google.colab import files
files.download("duka_sales.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])

df.head()

,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


# Duka Sales Exploration

## Executive Summary

1. The dataset contains 1,000 sales orders covering the period from January 1 to June 29, 2026.
2. Total revenue across all orders is KSh 71,792,500, with Nairobi contributing the highest branch revenue at KSh 26,106,100.
3. Laptops generated the highest product revenue at KSh 41,535,000, followed by Tablets at KSh 19,348,000.
4. Weekday orders generated KSh 51,856,200 from 723 orders, while weekend orders generated KSh 19,936,300 from 277 orders.
5. April was the highest-revenue month at KSh 13,518,300, while June was the lowest at KSh 10,903,400, a difference of about 24.0%.

## First-Five-Minutes Data Inspection

Before analysis, I inspected the dataset structure, data types, missing values, date range, and unique categorical values.

In [5]:
print("Shape:")
print(df.shape)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

print("\nDate range:")
print(df["order_date"].min(), "to", df["order_date"].max())

print("\nUnique categorical values:")
for column in df.select_dtypes(include="object").columns:
    print(column, ":", df[column].unique())

Shape:
(1000, 7)

Data types:
order_id               int64
order_date    datetime64[ns]
branch                object
product               object
quantity               int64
channel               object
unit_price             int64
dtype: object

Missing values:
order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64

Date range:
2026-01-01 00:00:00 to 2026-06-29 00:00:00

Unique categorical values:
branch : ['Nakuru' 'Nairobi' 'Mombasa' 'Eldoret' 'Kisumu']
product : ['Phone' 'Laptop' 'Tablet' 'Headphones' 'Charger']
channel : ['In-store' 'Online' 'Phone order']


### Inspection Findings

The dataset contains 1,000 rows and 7 columns. The order_date column is stored as a datetime field, while quantity and unit_price are numeric fields and branch, product, and channel are categorical fields. There are no missing values in the dataset. The sales period runs from January 1, 2026 to June 29, 2026. The dataset contains five branches, five products, and three sales channels.

In [6]:
df["revenue"] = df["unit_price"] * df["quantity"]

df["month"] = df["order_date"].dt.month

df["weekday"] = df["order_date"].dt.day_name()

df["is_weekend"] = df["order_date"].dt.dayofweek >= 5

df["price_band"] = np.where(
    df["unit_price"] >= 28000,
    "Premium",
    "Value"
)

df.head()

,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000,45000,6,Wednesday,False,Value
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000,260000,2,Monday,False,Premium
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000,140000,1,Monday,False,Premium
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500,12500,4,Sunday,True,Value
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500,7500,3,Saturday,True,Value


### Derived Columns

I created five derived columns for the analysis. Revenue is calculated as unit price multiplied by quantity. Month and weekday are extracted from the order date, while is_weekend identifies Saturday and Sunday orders. Price_band classifies products priced at KSh 28,000 or more as Premium and the remaining products as Value.

## Task 3 — Branch Performance

This section compares total revenue and number of orders across the five Duka branches.

In [7]:
branch_summary = (
    df.groupby("branch")
      .agg(
          total_revenue=("revenue", "sum"),
          number_of_orders=("revenue", "size")
      )
      .sort_values("total_revenue", ascending=False)
)

branch_summary

,total_revenue,number_of_orders
branch,,
Nairobi,26106100,376
Mombasa,14003700,186
Nakuru,11970400,162
Kisumu,10868100,159
Eldoret,8844200,117


In [8]:
top_branch = branch_summary.index[0]
top_branch_revenue = branch_summary.iloc[0]["total_revenue"]

lowest_branch = branch_summary.index[-1]
lowest_branch_revenue = branch_summary.iloc[-1]["total_revenue"]

print("Highest-revenue branch:", top_branch)
print("Highest branch revenue: KSh", f"{top_branch_revenue:,.0f}")
print()
print("Lowest-revenue branch:", lowest_branch)
print("Lowest branch revenue: KSh", f"{lowest_branch_revenue:,.0f}")

Highest-revenue branch: Nairobi
Highest branch revenue: KSh 26,106,100

Lowest-revenue branch: Eldoret
Lowest branch revenue: KSh 8,844,200


### Task 3 Answer

The branch summary shows the total revenue and number of orders for each branch. The branches are ranked from highest to lowest revenue in the table above. The highest-revenue branch is identified from the first row of the sorted table, while the lowest-revenue branch is identified from the last row. These results help show how sales performance differs across the Duka branches.

## Task 4 — Top Products

This section identifies the top three products by total revenue and by total units sold.

In [9]:
top_products_revenue = (
    df.groupby("product")["revenue"]
      .sum()
      .sort_values(ascending=False)
      .head(3)
)

print("Top 3 products by revenue:")
print(top_products_revenue)

Top 3 products by revenue:
product
Laptop    41535000
Tablet    19348000
Phone      8880000
Name: revenue, dtype: int64


In [10]:
top_products_units = (
    df.groupby("product")["quantity"]
      .sum()
      .sort_values(ascending=False)
      .head(3)
)

print("Top 3 products by units sold:")
print(top_products_units)

Top 3 products by units sold:
product
Tablet    691
Laptop    639
Phone     592
Name: quantity, dtype: int64


### Task 4 Answer

The analysis identifies the three products generating the highest total revenue and the three products with the highest number of units sold. The revenue ranking is based on the sum of the revenue column, while the units ranking is based on the sum of the quantity column. The two rankings show whether the products generating the most money are also the products selling the most units.

## Task 5 — Weekend vs Weekday Sales

This section compares the number of orders, total revenue, and average order revenue between weekdays and weekends.

In [11]:
weekend_summary = df.groupby("is_weekend").agg(
    number_of_orders=("revenue", "size"),
    total_revenue=("revenue", "sum"),
    average_order_revenue=("revenue", "mean")
)

weekend_summary.index = weekend_summary.index.map({
    False: "Weekday",
    True: "Weekend"
})

weekend_summary

,number_of_orders,total_revenue,average_order_revenue
is_weekend,,,
Weekday,723,51856200,71723.651452
Weekend,277,19936300,71972.202166


In [12]:
weekday_revenue = weekend_summary.loc["Weekday", "total_revenue"]
weekend_revenue = weekend_summary.loc["Weekend", "total_revenue"]

weekday_orders = weekend_summary.loc["Weekday", "number_of_orders"]
weekend_orders = weekend_summary.loc["Weekend", "number_of_orders"]

print("Weekday orders:", weekday_orders)
print("Weekday revenue: KSh", f"{weekday_revenue:,.0f}")
print()
print("Weekend orders:", weekend_orders)
print("Weekend revenue: KSh", f"{weekend_revenue:,.0f}")

Weekday orders: 723
Weekday revenue: KSh 51,856,200

Weekend orders: 277
Weekend revenue: KSh 19,936,300


### Task 5 Answer

The comparison shows how sales differ between weekdays and weekends. Weekday and weekend orders are compared using the number of orders, total revenue, and average order revenue. The table above provides the exact values for each group and shows whether weekends or weekdays generated more sales revenue.

## Task 6 — Monthly Revenue Analysis

In [13]:
monthly_revenue = (
    df.groupby("month")["revenue"]
      .sum()
      .sort_values(ascending=False)
)

best_month = monthly_revenue.idxmax()
best_revenue = monthly_revenue.max()

worst_month = monthly_revenue.idxmin()
worst_revenue = monthly_revenue.min()

percentage_difference = (
    (best_revenue - worst_revenue) / worst_revenue
) * 100

month_names = {
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June"
}

print("Monthly revenue:")
print(monthly_revenue)

print()
print("Highest-revenue month:", month_names[best_month])
print("Highest monthly revenue: KSh", f"{best_revenue:,.0f}")

print()
print("Lowest-revenue month:", month_names[worst_month])
print("Lowest monthly revenue: KSh", f"{worst_revenue:,.0f}")

print()
print("Percentage difference:", f"{percentage_difference:.1f}%")

Monthly revenue:
month
4    13518300
3    12396800
1    11786600
2    11772000
5    11415400
6    10903400
Name: revenue, dtype: int64

Highest-revenue month: April
Highest monthly revenue: KSh 13,518,300

Lowest-revenue month: June
Lowest monthly revenue: KSh 10,903,400

Percentage difference: 24.0%


### Answer

The month with the highest revenue was **[highest month]**, generating **KSh [amount]**. The month with the lowest revenue was **[lowest month]**, generating **KSh [amount]**. The difference between the highest and lowest monthly revenue was **[percentage]%**.

In [14]:
import os

# Create the outputs folder
os.makedirs("outputs", exist_ok=True)

# Filter the data
premium_online_coast_capital = df[
    (df["price_band"] == "Premium") &
    (df["channel"] == "Online") &
    (df["branch"].isin(["Nairobi", "Mombasa"]))
].copy()

# Calculate summary information
number_of_orders = len(premium_online_coast_capital)
total_revenue = premium_online_coast_capital["revenue"].sum()

# Save the filtered data
premium_online_coast_capital.to_csv(
    "outputs/premium_online_coast_capital.csv",
    index=False
)

print("Number of orders:", number_of_orders)
print("Total revenue: KSh", f"{total_revenue:,.0f}")
print("File saved successfully.")

Number of orders: 83
Total revenue: KSh 10,218,000
File saved successfully.


In [15]:
df.to_csv("duka_sales_analysis_data.csv", index=False)

print("Excel data file created successfully.")

Excel data file created successfully.
